# TKCE — 5-dataset clean benchmark (view ablation)

**What this runs:** the three view configs — `x` (raw only), `x+tree` (raw + RF split bits), `x+tree+deep` (FULL, adds the TabResNet extractor) — on **5 clean datasets** from the Grinsztajn suite, none of which are on TabReD's leak list:

| task | dataset |
|---|---|
| 361062 | pol |
| 361063 | house_16H |
| 361065 | MagicTelescope |
| 361277 | california |
| 361055 | credit |

Defaults: OOB-honest encoding, 2-member ensembles, 400 epochs, dropout 0.3 + L1 5e-5 + wd 1e-3, lr 3e-4.

### How to run
1. Runtime -> **GPU** (A100).
2. **Run all** (~1.5–2.5 h; shorten by editing cell 4: fewer tasks, `--epochs 300`, or `--ensemble 1`).

### What to look for (per dataset, in the final summary table)
- **x vs ceiling** = the genuine tree advantage on that dataset
- **x+tree vs x** = does the tree encoding help honestly?
- **FULL vs x+tree** = does the deep view add anything (on eye_movements it didn't)?

Optional (cell 4, commented): add `--encodings infold,oob` to also get the naive-vs-honest comparison — the paper's key table — at ~2x runtime.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> set Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 4 · RUN THE 5-DATASET SUITE
# (to also run the naive encoding for the infold-vs-oob paper table, use:
#  --encodings infold,oob   — roughly doubles the runtime)
!python -u run_clean_suite.py --encodings oob --epochs 400 --ensemble 2 --device auto

In [ ]:
# 5 · re-print the summary table + CSV
import pandas as pd
df = pd.read_csv('results/fusion/clean/summary.csv')
print(df.to_string(index=False))

In [ ]:
# 6 · show every figure
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/fusion/clean/*/*.png')):
    print('==', f, '==')
    display(Image(f))

In [ ]:
# 7 · download everything as one zip
import shutil
from google.colab import files
shutil.make_archive('clean_suite_results', 'zip', 'results/fusion/clean')
files.download('clean_suite_results.zip')